# Num. Epoch = 5

## Px - scaling 0

In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 40463.34it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 7.836097
mean ε  : 4.875823
median ε: 4.313324
std ε   : 2.440067
min ε   : 0.473779


In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 41258.76it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 0.879562
mean ε  : 0.600552
median ε: 0.677747
std ε   : 0.246374
min ε   : 0.065592


## Px - scaling 1

In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 41261.13it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 36.077882
mean ε  : 24.428476
median ε: 35.241316
std ε   : 15.718871
min ε   : 0.421725


In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 39634.27it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 1.387143
mean ε  : 0.745608
median ε: 0.699868
std ε   : 0.453946
min ε   : 0.028254


## Px - scaling 2

In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 40014.04it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 36.240018
mean ε  : 24.882420
median ε: 35.423721
std ε   : 15.010135
min ε   : 0.427526


In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 41691.42it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 1.579184
mean ε  : 0.766885
median ε: 0.761336
std ε   : 0.420826
min ε   : 0.027739


# Num. Epoch = 10

## Px - scaling 0

In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling0_ep10/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 40905.84it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 8.135086
mean ε  : 4.870637
median ε: 4.376365
std ε   : 2.486671
min ε   : 0.468641


In [ ]:
# ----------------------------------------------------------------------
# KL divergence
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
import torch
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling0_ep10/*.npy"):
    m = pattern.search(f)
    if not m: 
        continue
    key = tuple(map(int, m.groups()))  # (client_id, round)
    arr = np.load(f)
    t   = torch.from_numpy(arr).double()
    if "descriptor" in f:
        descriptors[key] = t
    elif "features" in f:
        features[key] = t
    elif "labels" in f:
        labels[key] = t

# Sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Flatten helper: (n, c, h, w) -> (n, c*h*w)
# ----------------------------------------------------------------------
def flatten(tensor):
    return tensor.view(tensor.size(0), -1) if tensor.dim() > 2 else tensor

# ----------------------------------------------------------------------
# Build Gaussian summaries (mean, var) per client-round with eps flooring
# ----------------------------------------------------------------------
eps = 1e-6
gaussians = {}
for k in descriptors:
    X   = flatten(features[k])
    mu  = X.mean(dim=0)                               # (d,)
    var = X.var(dim=0, unbiased=False) + eps          # floor to eps
    # optionally: var = var.clamp(min=eps)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Symmetric KL divergence between diagonal Gaussians (stable)
# ----------------------------------------------------------------------
def kl_diag(mu1, var1, mu2, var2):
    # ½ ∑ [log(var2/var1) + (var1 + (μ1-μ2)²)/var2 − 1]
    return 0.5 * torch.sum(
        torch.log(var2 / var1)
        + (var1 + (mu1 - mu2).pow(2)) / var2
        - 1.0
    )

def symm_kl(mu1, var1, mu2, var2):
    return kl_diag(mu1, var1, mu2, var2) + kl_diag(mu2, var2, mu1, var1)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round)
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # profile distance
    d1, d2 = descriptors[k1], descriptors[k2]
    profile_dist = torch.norm(d1 - d2)

    # reference distance (symmetric KL)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist  = symm_kl(mu1, var1, mu2, var2)

    epsilons.append((profile_dist - ref_dist).abs().item())

eps = torch.tensor(epsilons)

print("\nKL divergence ε statistics over {:d} pairs:".format(len(eps)))
print("-" * 40)
print(f"max ε   : {eps.max():.6f}")
print(f"mean ε  : {eps.mean():.6f}")
print(f"median ε: {eps.median():.6f}")
print(f"std ε   : {eps.std():.6f}")
print(f"min ε   : {eps.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:37<00:00, 1189.63it/s]


KL divergence ε statistics over 44850 pairs:
----------------------------------------
max ε   : 3424284.250000
mean ε  : 906361.062500
median ε: 830314.062500
std ε   : 647082.875000
min ε   : 84.953590


In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling0_ep10/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 40953.57it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 0.899070
mean ε  : 0.596592
median ε: 0.692090
std ε   : 0.246765
min ε   : 0.060133


## Px - scaling 1

In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling1_ep10/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 41704.79it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 36.063942
mean ε  : 24.189166
median ε: 35.310966
std ε   : 15.844756
min ε   : 0.403378


In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling1_ep10/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 40096.90it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 1.350097
mean ε  : 0.697537
median ε: 0.685764
std ε   : 0.427148
min ε   : 0.028193


## Px - scaling 2

In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling2_ep10/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 43154.35it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 36.272146
mean ε  : 25.215731
median ε: 35.454286
std ε   : 14.911257
min ε   : 0.412515


In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling2_ep10/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 37816.52it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 1.519817
mean ε  : 0.742564
median ε: 0.713812
std ε   : 0.405239
min ε   : 0.029636


.

.

.

.

.

.

. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

.

.

.

.

.

.



# Num. Epoch = 5


## Py - scaling 0

In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Py_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:00<00:00, 57551.65it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 6.521107
mean ε  : 3.562209
median ε: 3.394144
std ε   : 1.792107
min ε   : 0.366779


In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Py_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 38744.31it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 9.923859
mean ε  : 6.210537
median ε: 6.982712
std ε   : 3.824700
min ε   : 0.005944


## Py - scaling 1


In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Py_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:00<00:00, 59696.44it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 6.361378
mean ε  : 3.712219
median ε: 3.641250
std ε   : 1.520556
min ε   : 0.368475


In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Py_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 41329.20it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 9.755121
mean ε  : 6.562941
median ε: 6.886243
std ε   : 3.222152
min ε   : 0.004628


## Py - scaling 2

In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Py_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:00<00:00, 51154.28it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 6.995150
mean ε  : 3.818400
median ε: 3.908250
std ε   : 1.465198
min ε   : 0.422966


In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Py_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 39476.43it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 9.525443
mean ε  : 7.236615
median ε: 9.390459
std ε   : 3.060093
min ε   : 0.006373


.

.

.

.

.

.

. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

.

.

.

.

.

.

# Num. Epoch = 5

## P(X|Y) - scaling 0

In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pxy_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 42056.69it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 27.645698
mean ε  : 14.965917
median ε: 23.402558
std ε   : 11.436934
min ε   : 0.196417


In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pxy_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 41508.81it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 6.723670
mean ε  : 4.144692
median ε: 5.022936
std ε   : 2.281440
min ε   : 0.121112


## P(X|Y) - scaling 1

In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pxy_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 37212.31it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 29.438110
mean ε  : 17.920535
median ε: 25.297046
std ε   : 11.796046
min ε   : 0.000067


In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pxy_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 38272.51it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 5.444935
mean ε  : 3.232773
median ε: 3.425298
std ε   : 1.646229
min ε   : 0.097945


## P(X|Y) - scaling 2

In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pxy_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 40826.31it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 30.041569
mean ε  : 17.638544
median ε: 26.972675
std ε   : 12.153569
min ε   : 0.401694


In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pxy_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 36787.46it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 3.221727
mean ε  : 2.006825
median ε: 2.456455
std ε   : 0.990996
min ε   : 0.072391


.

.

.

.

.

.

. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

.

.

.

.

.

.

# Num. Epoch = 5

## P(Y|X) - scaling 0

In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pyx_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:00<00:00, 49208.01it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 1.046536
mean ε  : 0.514786
median ε: 0.505534
std ε   : 0.105381
min ε   : 0.189708


In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pyx_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 37714.48it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 0.572246
mean ε  : 0.382763
median ε: 0.400376
std ε   : 0.078306
min ε   : 0.087630


## P(Y|X) - scaling 1

In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pyx_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:00<00:00, 50119.81it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 1.016012
mean ε  : 0.468606
median ε: 0.456030
std ε   : 0.113019
min ε   : 0.149197


In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pyx_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 40897.37it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 0.593892
mean ε  : 0.428944
median ε: 0.456223
std ε   : 0.086819
min ε   : 0.071587


## P(Y|X) - scaling 2

In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pyx_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:00<00:00, 55720.59it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 0.996047
mean ε  : 0.447326
median ε: 0.436051
std ε   : 0.107942
min ε   : 0.126462


In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pyx_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 40011.91it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 0.625911
mean ε  : 0.450223
median ε: 0.469491
std ε   : 0.079728
min ε   : 0.068781
